# 4장 2강: 교차검증 전략과 데이터 누수 방지

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 불균형 분류 문제에서 적절한 교차검증 방법을 선택하고, 데이터 누수를 방지하는 방법을 연습합니다.
이번 실습은 거래 간 시간 의존성을 고려하지 않는 분류 연습으로 StratifiedKFold를 사용하며, 실제 미래 거래 예측을 평가할 때는 시간 순서를 보존해 분할한다.

- KFold 교차검증의 기본 원리를 확인합니다.
- 불균형 분류에서 StratifiedKFold가 필요한 이유를 확인합니다.
- KFold / StratifiedKFold / TimeSeriesSplit의 선택 기준을 구분합니다.
- 잘못된 전처리 순서가 데이터 누수를 만들 수 있음을 이해합니다.
- `Pipeline`을 이용해 전처리와 모델 학습을 교차검증 안에서 안전하게 연결합니다.
- SMOTE를 사용할 때 `imbalanced-learn Pipeline`을 이용해 Train Fold에만 적용되도록 구성합니다.

> 이 실습에서는 4장 2강 교안에 나온 교차검증, 데이터 누수, Pipeline 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습은 불균형 분류 문제이므로 `Class` 비율을 유지하는 교차검증 전략이 중요합니다.

## 실습 준비

1. 데이터를 불러옵니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 입력 특성 X로 사용합니다.
4. `Class`를 정답 y로 사용합니다.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

data = pd.read_csv("creditcard.csv")

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(y.value_counts().to_frame("count"))

print("\n[Class 비율]")
display(y.value_counts(normalize=True).to_frame("ratio"))

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727


---

# 필수 1. KFold와 StratifiedKFold 비교

## 배경

KFold는 데이터를 여러 Fold로 나누어 반복 평가합니다.

하지만 Credit Card Fraud처럼 클래스 비율이 크게 불균형한 데이터에서는 Fold마다 사기 거래 비율이 달라질 수 있습니다.

StratifiedKFold는 각 Fold에서 원본 데이터와 비슷한 클래스 비율을 유지하도록 분할합니다.

## 문제 1. KFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `KFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Fold의 Validation 데이터에서 `Class=1` 비율을 계산합니다.
3. Fold별 사기 거래 비율을 출력합니다.

### 확인 포인트

- KFold는 클래스 비율 자체를 유지하도록 설계된 방법이 아님을 이해했는가?
- Fold별 소수 클래스 비율이 조금씩 달라질 수 있음을 확인했는가?

In [4]:
# TODO
# KFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f'원본 사기 비율: {y.mean()}')

for i, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    y_val = y.iloc[val_idx]
    print(f'Fold {i}: 사기 비율 = {y_val.mean():.4f}, 사기 건수 = {y_val.sum()}')

원본 사기 비율: 0.001727485630620034
Fold 1: 사기 비율 = 0.0017, 사기 건수 = 98
Fold 2: 사기 비율 = 0.0016, 사기 건수 = 93
Fold 3: 사기 비율 = 0.0017, 사기 건수 = 95
Fold 4: 사기 비율 = 0.0018, 사기 건수 = 101
Fold 5: 사기 비율 = 0.0018, 사기 건수 = 105


## 문제 2. StratifiedKFold의 Fold별 클래스 비율을 확인하세요.

### 요구사항

1. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
2. 각 Validation Fold의 `Class=1` 비율을 계산합니다.
3. 원본 전체 데이터의 사기 거래 비율과 비교합니다.

### 확인 포인트

- 각 Fold의 클래스 비율이 원본과 비슷하게 유지되는가?
- 불균형 분류에서 StratifiedKFold가 적합한 이유를 설명할 수 있는가?

In [6]:
# TODO
# StratifiedKFold의 각 Validation Fold에서 Class=1 비율을 확인하세요.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'원본 사기 비율: {y.mean()}')

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    y_val = y.iloc[val_idx]
    print(f"Fold {i}: 사기 비율 = {y_val.mean():.4f}, 사기 건수 = {y_val.sum()}")

원본 사기 비율: 0.001727485630620034
Fold 1: 사기 비율 = 0.0017, 사기 건수 = 99
Fold 2: 사기 비율 = 0.0017, 사기 건수 = 99
Fold 3: 사기 비율 = 0.0017, 사기 건수 = 98
Fold 4: 사기 비율 = 0.0017, 사기 건수 = 98
Fold 5: 사기 비율 = 0.0017, 사기 건수 = 98


## 문제 3. 두 교차검증 방법으로 F1-score를 비교하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 사용합니다.
2. KFold로 5-Fold F1-score를 계산합니다.
3. StratifiedKFold로 5-Fold F1-score를 계산합니다.
4. 각 방법의 Fold별 점수, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 한 번의 점수가 아니라 여러 Fold의 평균과 편차를 확인했는가?
- Credit Card Fraud와 같은 불균형 분류에서는 어떤 방법이 더 안전한 기본 선택인지 설명할 수 있는가?

In [7]:
# TODO
# KFold와 StratifiedKFold의 F1-score 평균과 표준편차를 비교하세요.
model = LogisticRegression(max_iter=1000)

kf_scores = cross_val_score(model, X, y, cv=kf, scoring="f1")
skf_scores = cross_val_score(model, X, y, cv=skf, scoring="f1")

print("Fold별:", kf_scores.round(4))
print("평균:", kf_scores.mean().round(4), "/ 표준편차:", kf_scores.std().round(4))

print("\n[StratifiedKFold]")
print("Fold별:", skf_scores.round(4))
print("평균:", skf_scores.mean().round(4), "/ 표준편차:", skf_scores.std().round(4))

Fold별: [0.6829 0.7453 0.716  0.7108 0.7513]
평균: 0.7213 / 표준편차: 0.0249

[StratifiedKFold]
Fold별: [0.6909 0.7168 0.7073 0.7636 0.7614]
평균: 0.728 / 표준편차: 0.0294


### Q1. Credit Card Fraud 데이터에서는 KFold와 StratifiedKFold 중 어떤 방법을 우선 사용하는 것이 적절한가요?

**답변:**  StratifiedKFold를 우선 사용하는 게 적절하다. 사기 거래가 0.17%밖에 안 돼서, KFold로 나누면 Fold마다 사기 건수가 93~105건으로 달라졌다. StratifiedKFold는 모든 Fold를 98~99건으로 맞춰줘서 Fold끼리 같은 조건에서 평가할 수 있다. 이번에는 점수 차이가 크지 않았지만, 비율이 보장된다는 점에서 더 안전한 기본 선택이다.

### Q2. 다음 상황에서 어떤 교차검증 방법을 선택할지 작성하세요.

1. 일반적인 회귀 문제
2. 불균형 분류 문제
3. 시간 순서가 중요한 시계열 문제

**답변:**  
- 일반적인 회귀 문제는 KFold를 쓴다. 정답이 연속값이라 클래스 비율을 맞출 필요가 없다.
- 불균형 분류 문제는 StratifiedKFold를 쓴다. Fold마다 소수 클래스 비율을 유지해야 한다.
- 시계열 문제는 TimeSeriesSplit을 쓴다. 과거 데이터로 학습하고 미래 데이터로 검증해야 하므로, 섞으면 안 된다.

---

# 필수 2. Pipeline으로 데이터 누수 방지

## 배경

전처리를 전체 데이터에 먼저 `fit()`한 뒤 교차검증하면 Validation Fold의 정보가 전처리 기준에 포함될 수 있습니다.

교안에서는 이를 방지하기 위해 **전처리와 모델을 Pipeline으로 묶는 방법**을 사용합니다.

## 문제 1. StandardScaler와 Logistic Regression을 Pipeline으로 연결하세요.

### 요구사항

1. `StandardScaler()`와 `LogisticRegression(class_weight="balanced", max_iter=1000)`을 하나의 `Pipeline`으로 만듭니다.
2. `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`를 사용합니다.
3. `cross_val_score()`로 F1-score를 계산합니다.
4. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- 각 Fold의 Train 데이터에서만 scaler가 fit되는 구조인가?
- Validation Fold에는 Train에서 학습한 기준으로 transform만 적용되는가?
- 불균형 분류이므로 StratifiedKFold를 사용했는가?

In [8]:
# TODO
# StandardScaler + LogisticRegression을 Pipeline으로 연결하고
# StratifiedKFold로 F1-score를 계산하세요.
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000))
])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_score = cross_val_score(
    pipe,
    X,
    y,
    cv=skf,
    scoring='f1'
)
print(f'Fold별: {pipe_score}')
print(f'평균: {pipe_score.mean()}')
print(f'표준편차: {pipe_score.std()}')

Fold별: [0.12818645 0.12       0.11206897 0.11180124 0.11748634]
평균: 0.11790859991473175
표준편차: 0.006026403089505455


### Q3. Pipeline을 사용하면 스케일링 누수를 방지할 수 있는 이유는 무엇인가요?

**답변:**  Pipeline을 cross_val_score에 넣으면 Fold마다 Train 부분으로만 scaler를 fit하고, Validation 부분은 그 기준으로 transform만 한다. 그래서 Validation 데이터의 평균이나 표준편차가 scaler 학습에 들어가지 않는다. 전처리 순서를 사람이 직접 관리하지 않아도 자동으로 지켜지는 게 장점이다.

## 문제 2. SMOTE를 교차검증 안에서 안전하게 적용하세요.

### 배경

SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 Validation Fold와 매우 비슷한 합성 데이터가 Train에 포함될 수 있습니다.

교안에서는 `imbalanced-learn`의 Pipeline 안에 SMOTE를 넣어 각 Fold의 Train 부분에만 적용하도록 합니다.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. `LogisticRegression(max_iter=1000)`과 함께 `ImbPipeline`으로 연결합니다.
3. `StratifiedKFold`를 사용합니다.
4. `cross_val_score()`로 F1-score를 계산합니다.
5. Fold별 F1-score, 평균, 표준편차를 출력합니다.

### 확인 포인트

- SMOTE가 교차검증 밖에서 전체 데이터에 적용되지 않았는가?
- 각 Fold의 Train 부분에만 SMOTE가 적용되는 구조인가?

In [9]:
# TODO
# SMOTE + LogisticRegression을 imbalanced-learn Pipeline으로 연결하세요.
pipe_sm = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000))
])

sm_scores = cross_val_score(
    pipe_sm,
    X,
    y,
    cv=skf,
    scoring='f1'
)
print(f'Fold별: {sm_scores}')
print(f'평균: {sm_scores.mean()}')
print(f'표준편차: {sm_scores.std()}')

Fold별: [0.12258953 0.11071429 0.10650888 0.10439883 0.10689869]
평균: 0.11022204299107201
표준편차: 0.006510594105889241


### Q4. SMOTE를 전체 데이터에 먼저 적용한 뒤 교차검증하면 왜 문제가 될 수 있나요?

**답변:** SMOTE는 기존 사기 데이터 사이에 새로운 데이터를 만든다. 그래서 전체 데이터에 먼저 적용하면, Validation에 들어갈 진짜 사기 데이터를 바탕으로 만든 비슷한 합성 데이터가 Train에 들어갈 수 있다. 그러면 모델이 사실상 정답을 미리 본 것과 비슷해져서 점수가 실제보다 높게 나온다. 또 Validation에도 합성 데이터가 섞이게 돼서 실제 데이터로 평가하는 게 아니게 된다.

---

# 과제 1. 데이터 특성에 맞는 CV와 누수 없는 Pipeline 구성

## 배경

필수 실습에서는 불균형 분류에 적합한 StratifiedKFold와 Pipeline을 직접 사용했습니다.

이번 과제에서는 같은 Credit Card Fraud 데이터에서 **누수 없는 평가 흐름을 스스로 구성하고 설명**합니다.

> 과제는 필수 실습에서 사용한 구성 요소를 다시 연결하는 수준입니다.

## 요구사항

1. `StandardScaler()`를 사용합니다.
2. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
3. 두 단계를 `Pipeline`으로 연결합니다.
4. Credit Card Fraud 데이터 특성에 맞는 교차검증 방법을 직접 선택합니다.
5. `n_splits=5`를 사용합니다.
6. `cross_val_score(..., scoring="f1")`로 성능을 평가합니다.
7. Fold별 F1-score, 평균, 표준편차를 출력합니다.
8. 왜 해당 교차검증 방법을 선택했는지 설명합니다.
9. 다음 잘못된 순서를 데이터 누수 관점에서 설명합니다.

```text
전체 X로 StandardScaler.fit_transform()
        ↓
교차검증 수행
```

10. 시계열 데이터라면 같은 방법을 그대로 사용하면 안 되는 이유와 어떤 CV를 선택해야 하는지 작성합니다.

11. `build_pipeline(model)` 함수를 작성하여 `StandardScaler`와 전달받은 모델을 연결한 Pipeline을 반환합니다. 이 함수 안에서는 데이터를 학습하거나 변환하지 않습니다.
12. `evaluate_pipeline(X, y, model, cv, scoring="f1")` 함수를 작성합니다. 11번 함수를 이용해 Pipeline을 만들고 교차검증한 뒤, Fold별 점수·평균·표준편차를 반환합니다. 모델·CV·평가지표를 함수 인자로 교체할 수 있어야 합니다.
13. 12번 함수를 실제 호출하여 요구사항 7번 결과를 출력합니다. CV에서 데이터를 섞는 경우 `shuffle=True, random_state=42`를 지정하고, 모델 등 무작위성이 있는 구성 요소도 가능한 경우 `random_state=42`로 고정합니다.
14. 함수의 재사용 방법과 재현성 설정을 설명하고 Q5~Q8에 답합니다. 별도 `.py` 모듈 제작이나 다른 모델의 추가 학습은 요구하지 않습니다.

### 확인 포인트

- 불균형 분류 특성에 맞는 CV를 선택했는가?
- Pipeline이 전처리 누수를 방지하는 구조임을 설명했는가?
- 평균뿐 아니라 Fold별 점수와 표준편차도 확인했는가?
- TimeSeriesSplit의 사용 상황을 구분할 수 있는가?
- 함수 인자로 모델·CV·평가지표를 바꿀 수 있는가?
- 함수 호출로 전처리·학습·평가가 연결되고 점수가 반환되는가?
- 재현성을 위한 난수 설정을 명시했는가?


In [10]:
# TODO
# build_pipeline(model)과 evaluate_pipeline(X, y, model, cv, scoring)를 작성하세요.
# 적절한 5-Fold CV와 모델을 전달하고 F1 점수·평균·표준편차를 출력하세요.

def build_pipeline(model):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])
    return pipe


def evaluate_pipeline(X, y, model, cv, scoring="f1"):
    pipe = build_pipeline(model)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring=scoring)
    return scores, scores.mean(), scores.std()


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)

scores, mean_score, std_score = evaluate_pipeline(X, y, model, cv, scoring="f1")

print(f"Fold별 F1: {scores}")
print(f"평균: {mean_score}")
print(f"표준편차: {std_score}")


Fold별 F1: [0.12818645 0.12       0.11206897 0.11180124 0.11748634]
평균: 0.11790859991473175
표준편차: 0.006026403089505455


### Q5. 이번 Credit Card Fraud 데이터에서 선택한 CV와 그 이유를 작성하세요.

**답변:**  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)를 선택했다. 사기 거래가 0.17%뿐인 불균형 분류라서, Fold마다 사기 비율을 원본과 같게 유지해야 한다. 실제로 확인해보니 모든 Fold에 사기가 98~99건씩 들어갔다.

### Q6. 전체 데이터에 StandardScaler를 먼저 fit하면 어떤 정보가 Validation Fold에서 Train 쪽으로 새어 들어갈 수 있나요?

**답변:**  Validation Fold 데이터의 평균과 표준편차가 새어 들어간다. 전체 데이터로 scaler를 fit하면 평균과 표준편차를 Validation 데이터까지 포함해서 계산하므로, Train 데이터를 변환하는 기준에 Validation 정보가 섞이게 된다.

### Q7. 작성한 함수를 다른 모델·CV·평가지표에 재사용하려면 어떤 인자를 바꾸면 되나요?

**답변:** evaluate_pipeline의 model, cv, scoring 인자를 바꾸면 된다. 예를 들어 다른 모델은 model=RandomForestClassifier(random_state=42), 시계열이면 cv=TimeSeriesSplit(n_splits=5), 다른 지표는 scoring="recall"이나 scoring="average_precision"으로 바꾸면 된다.


### Q8. 재현성을 위해 어떤 설정을 고정했으며, Pipeline 생성 함수 안에서 전체 데이터로 전처리를 미리 학습하면 안 되는 이유는 무엇인가요?

**답변:** 재현성을 위해 StratifiedKFold에 shuffle=True, random_state=42를 넣었고, LogisticRegression에도 random_state=42를 넣었다. 이렇게 하면 다시 돌려도 Fold가 똑같이 나뉘어 같은 결과가 나온다.
build_pipeline 안에서 전체 데이터로 전처리를 미리 fit하면 Validation 정보까지 들어간 scaler가 만들어진다. 그러면 Pipeline을 써도 누수를 막을 수 없다. 그래서 함수에서는 Pipeline 틀만 만들고, 실제 fit은 cross_val_score가 Fold마다 Train 데이터로만 하게 해야 한다.


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 교차검증을 사용하는 이유는 무엇인가요?
2. KFold와 StratifiedKFold의 가장 큰 차이는 무엇인가요?
3. 시계열 데이터에는 어떤 교차검증 방법이 적절한가요?
4. StandardScaler는 전체 데이터가 아니라 어디에서 fit되어야 하나요?
5. SMOTE는 각 Fold의 Train / Validation 중 어디에만 적용되어야 하나요?
6. Pipeline이 데이터 누수 방지에 도움이 되는 이유는 무엇인가요?

- 데이터를 한 번만 나눠서 평가하면 어떻게 나눴느냐에 따라 점수가 달라질 수 있어서, 여러 번 나눠 평균과 편차를 보고 더 믿을 수 있는 성능을 확인하기 위해서다.
- KFold는 그냥 무작위로 나누고, StratifiedKFold는 각 Fold의 클래스 비율을 원본과 같게 맞춰서 나눈다.
- TimeSeriesSplit이 적절하다.
- 각 Fold의 Train 데이터에서만 fit되어야 한다.
- Train Fold에만 적용해야 한다.
- 전처리와 모델을 하나로 묶어서, 교차검증 때 Fold마다 Train에서만 fit하고 Validation에는 transform만 하도록 자동으로 처리해주기 때문이다.